# cuda-torso + GBDT booster — Google Colab (GPU)

Runs the leaderboard-winning **cuda-torso engine** with an additive **GBDT
front-booster** (this work's novelty), plus a `--no_gbdt` **control** for a clean
ablation. Everything is in `cuda_torso_gbdt.zip` (their `run.py` + `libeval.cu` +
`run_gbdt.py` + all three graphs).

**First: Runtime → Change runtime type → Hardware accelerator → GPU (T4).**
Then run the cells top to bottom. Colab usually gives **one** GPU, so the boosted
and control runs are done one at a time (see the two run cells).

In [ ]:
# 1) locate / upload cuda_torso_gbdt.zip and extract it
import os, glob, zipfile, shutil
cands = glob.glob('/content/**/cuda_torso_gbdt.zip', recursive=True)
if not cands:
    from google.colab import files
    print("Choose cuda_torso_gbdt.zip to upload:")
    up = files.upload()
    cands = ['/content/' + list(up.keys())[0]]
shutil.rmtree('/content/cuda', ignore_errors=True); os.makedirs('/content/cuda', exist_ok=True)
zipfile.ZipFile(cands[0]).extractall('/content/cuda')
ROOT = os.path.dirname(glob.glob('/content/cuda/**/run_gbdt.py', recursive=True)[0])
os.chdir(ROOT)
print("project:", ROOT, "| files:", sorted(os.listdir('.')))

In [ ]:
# 2) compile the cuda-torso evaluator -> libeval.so  (one time)
!cd /content/cuda/cuda-torso-main && nvcc -shared -Xcompiler -fPIC -o libeval.so libeval.cu && echo OK && ls -la libeval.so

In [ ]:
# 3) GPU + deps check (sklearn powers the GBDT booster)
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
try:
    import sklearn; print("sklearn", sklearn.__version__)
except ImportError:
    import subprocess; subprocess.run(["pip","install","-q","scikit-learn"]); import sklearn; print("installed sklearn", sklearn.__version__)
# how many GPUs? (Colab is usually 1)
print("GPU count:", torch.cuda.device_count())

## Smoke test — small-graph (confirm both modes behave)
Small is near-optimal, so the booster's gain is ~0 here; the point is to confirm
the engine reaches ~the optimum and `bonus-t` rises above 0 after gen 200 (proves
the booster is live). Stop the cell (⏹) after a few hundred generations.

In [ ]:
# boosted
!cd /content/cuda/cuda-torso-main && python3 run_gbdt.py --graph small-graph --gbdt_every 200 --seed 0

## The real run — medium or large (where there is headroom)
Colab gives one GPU, so run the **boosted** arm first (foreground cell below),
let it checkpoint, then run the **control** with the same seed. Both write
`submissions/<graph>/<score>.json` every 50 gens (filename = HVI; more negative =
better). NOTE: medium/large are ~6x / far denser than small, so from a random
start the first generations are slow (heavy fill-in) and speed up as the front
improves — give it time.

In [ ]:
# boosted: cuda-torso engine + GBDT front-booster
!cd /content/cuda/cuda-torso-main && python3 run_gbdt.py --graph medium-graph --gbdt_every 200 --seed 0

In [ ]:
# control: vanilla cuda-torso (identical engine, seed, budget; GBDT OFF) -- run after the boosted arm
!cd /content/cuda/cuda-torso-main && python3 run_gbdt.py --graph medium-graph --no_gbdt --seed 0

### (Optional) run both at once in the background
Only if Colab gave you **2 GPUs** (rare). If `torch.cuda.device_count()` was 1,
skip this and use the two foreground cells above instead.

In [ ]:
import subprocess, os
os.chdir('/content/cuda/cuda-torso-main'); G="medium-graph"
subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 python3 run_gbdt.py --graph {G} --gbdt_every 200 --seed 0 > boosted.log 2>&1", shell=True)
subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 python3 run_gbdt.py --graph {G} --no_gbdt   --seed 0 > control.log  2>&1", shell=True)
print("launched boosted (GPU0) + control (GPU1)")

In [ ]:
# monitor (works for the background-launch cell)
!echo '== BOOSTED ==' && tail -6 /content/cuda/cuda-torso-main/boosted.log 2>/dev/null
!echo '== CONTROL ==' && tail -6 /content/cuda/cuda-torso-main/control.log 2>/dev/null
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader

## Collect the best front
Most-negative filename in `submissions/<graph>/` is the best. Download it and
re-score locally with `tools/portfolio.py` before quoting any number.

In [ ]:
import glob, os, shutil
G="medium-graph"   # set to the graph you ran
subs = glob.glob(f'submissions/{G}/*.json')
best = min(subs, key=lambda f:int(os.path.basename(f).split('.')[0])) if subs else None
print("checkpoints:", len(subs), "| best:", best)
if best:
    shutil.copy(best, f'/content/{G}_best.json')
    try:
        from google.colab import files; files.download(f'/content/{G}_best.json')
    except Exception: pass
print("leaderboard targets: small -1,829,919 | medium -1,745,122 | large -5,493,062")